# Why does CUDA graph capture fail? (issue #122)

Capture of the collocation exact-Hessian region fails with:

```
CUDA error: operation failed due to a previous error during capture
cudaErrorStreamCaptureInvalidated
```

That names the **symptom, not the offending operation** -- CUDA reports that the
stream was already invalidated, not which call did it. So this bisects instead of guessing.

**This notebook is deliberately tiny: ~3 minutes.** A 1-day horizon (72 segments rather than
360) and 2 IPOPT iterations, purely to construct the Hessian closures with real inputs. The
capture experiments then run on those closures directly and take seconds each. Capture validity
is a property of *which operations run*, not of how many segments they run over, so the small
problem tests the same thing.

### What it tries

The region has two independent halves:

| piece | what it does | extra machinery |
|---|---|---|
| `_curvature` | per-link constraint curvature | `vmap(hessian(...))` |
| `_obj_curvature` | per-segment objective curvature | same, **plus** `composer.F_aug` |
| `_hess_core` | both, plus einsums and packing | everything |

Each is captured alone, under two `capture_error_mode` settings:

* **`"global"`** -- the default, strictest.
* **`"relaxed"`** -- tolerates some operations that would otherwise invalidate the stream. If
  `relaxed` succeeds where `global` fails, the offender is a *synchronisation* rather than an
  illegal API call, and relaxed mode may itself be the fix.

**Not used: `CUDA_LAUNCH_BLOCKING=1`.** It forces a device sync after every kernel, which would
invalidate capture on its own and make every arm fail for the wrong reason.

### Reading it

* `curvature` captures, `obj_curvature` does not -> the culprit is in `composer.F_aug`, i.e.
  a component's `forward`, and the next step is to bisect the components.
* Both fail -> the problem is structural to `torch.func` transforms under capture, and the
  approach is likely dead for this region.
* Both succeed alone but `core` fails -> the culprit is in the einsum/packing glue, which is
  the easiest case to fix.
* `relaxed` succeeds anywhere -> a sync is the cause, and it is worth locating.

### Why a canary

A failed capture can leave the CUDA context unable to capture **anything** afterwards, so
every later attempt reports the same *"previous error during capture"* and one failure
masquerades as many. The first version of this notebook had exactly that flaw: it ran the
solve with the graph enabled, which failed first and poisoned everything after it.

So: the solve now runs with the graph **off**, and a trivial capture (`a @ a`) is checked
before the first attempt and after every one. Once the canary dies, the remaining rows are
void and the run says so rather than reporting them as negative results.

**Setup**: Runtime > Change runtime type > **GPU**, then Run all.

In [ ]:
# Quieten the translator's per-type namespace warnings.  They are a known
# issue (#114), they number in the hundreds per model build, and with several
# model builds per A/B they bury the results completely.
import warnings

warnings.filterwarnings("ignore", message="Failed to parse namespace")
warnings.filterwarnings("ignore", message="Failed to parse ontology namespace")
warnings.filterwarnings("ignore", message='Neither "df", "filename", nor "uuid"')
# --- Setup (Colab-aware) ---------------------------------------------------
# Needs TWIN4BUILD_HESS_CHUNK_DIV, added alongside this notebook.
TWIN4BUILD_REF = "feature/issue-122/cuda-graph-hessian"

try:
    import twin4build as tb
except ImportError:
    # subprocess rather than %pip so the ref interpolates unambiguously.
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         f"git+https://github.com/JBjoernskov/Twin4Build.git@{TWIN4BUILD_REF}"],
        check=True,
    )
    import twin4build as tb

# Fail loudly now rather than 30 minutes in.
import inspect

import twin4build.estimator._transcription as _tr

_src = inspect.getsource(_tr)
_missing = [n for n in ("TWIN4BUILD_HESS_CHUNK_DIV", "exact_hessian",
                        "boundary_state_init") if n not in _src]
# A re-run in a live session does NOT reinstall (the import above succeeds), so
# without this check you would silently measure whatever build is already
# present.  capture_status() is the newest symbol, so its absence means stale.
try:
    from twin4build.estimator._cuda_graph import capture_status  # noqa: F401
except Exception:
    _missing.append("capture_status (stale install -- see below)")
if _missing:
    # Reaching here after a forced reinstall means the OLD module is still
    # loaded in this session -- pip cannot replace an already-imported module.
    raise RuntimeError(
        "The installed twin4build is missing: " + ", ".join(_missing)
        + f"""
Installed at: {tb.__file__}
Install a ref that has these, then restart the runtime:
    pip install -q --force-reinstall --no-deps git+https://github.com/JBjoernskov/Twin4Build.git@{TWIN4BUILD_REF}
    (Colab: Runtime > DISCONNECT AND DELETE RUNTIME, then Run all.
     "Restart session" alone is NOT enough -- the stale package survives it.)"""
    )

import functools
import os
import time

import numpy as np
import pandas as pd
import torch

DEVICES = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])


def hardware():
    import platform
    import re

    cpu = platform.processor() or platform.machine()
    try:
        with open("/proc/cpuinfo") as fh:
            for line in fh:
                if line.lower().startswith("model name"):
                    cpu = line.split(":", 1)[1].strip()
                    break
    except OSError:
        pass
    ram = None
    try:
        with open("/proc/meminfo") as fh:
            ram = int(re.search(r"[0-9]+", fh.readline()).group()) / 1e6
    except OSError:
        pass
    gpu = fp64_ratio = None
    if torch.cuda.is_available():
        pr = torch.cuda.get_device_properties(0)
        gpu = f"{pr.name} ({pr.total_memory / 1e9:.0f} GB, sm_{pr.major}{pr.minor})"
        # Datacenter parts (A100/V100, sm_70/80/90) run fp64 at 1/2 of fp32;
        # consumer parts at 1/32-1/64.  Recorded because it was the first
        # hypothesis for the exact-Hessian result and it needs to stay visible.
        fp64_ratio = "1/2 (datacenter)" if pr.major in (7, 8, 9) and pr.minor == 0 \
            else "1/32-1/64 (consumer) -- suspect"
    return {"cpu": cpu, "cores": os.cpu_count(),
            "torch_threads": torch.get_num_threads(),
            "ram_gb": round(ram, 1) if ram else None,
            "gpu": gpu, "fp64": fp64_ratio, "torch": torch.__version__}


HW = hardware()
for k, v in HW.items():
    print(f"{k:14s} {v}")
if not torch.cuda.is_available():
    print("\nNO GPU -- Part A still works; the GPU comparisons will be skipped.")

In [ ]:
import datetime
import importlib.util as _ilu
import pathlib

from dateutil import tz

import twin4build.examples.utils as utils
from twin4build.utils.rgetattr import rgetattr

# `twin4build/examples/full_workflow_example/` (packaged CSVs) is a PACKAGE that
# shadows the module `full_workflow_example.py`, so load the module by path.
import twin4build.examples as _ex_pkg

_p = pathlib.Path(_ex_pkg.__file__).parent / "full_workflow_example.py"
_spec = _ilu.spec_from_file_location("_fwe_runtime", _p)
_mod = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
fcn = _mod.fcn

STEP = 1200
START = [datetime.datetime(2023, 12, 2, tzinfo=tz.gettz("Europe/Copenhagen"))]
END = [datetime.datetime(2023, 12, 7, tzinfo=tz.gettz("Europe/Copenhagen"))]
N_WARMUP = 20


def build_model(device="cpu", dtype=torch.float64, tag="prof"):
    m = tb.Model(id=f"{tag}_{device}")
    m.load(semantic_model_filename=utils.get_path(
        ["estimator_example", "one_room_example_model.xlsm"]), fcn=fcn)
    m.to(device, dtype)
    return m


def build_parameters(model):
    c = model.components
    space, heater = c["office"], c["office_space_heater"]
    hc, cc = c["office_temperature_heating_controller"], c["office_co2_controller"]
    valve = c["office_space_heater_valve"]
    sup, exh = c["office_supply_damper"], c["office_exhaust_damper"]
    occ, wall = c["office_occupancy"], c["office_boundary_wall"]
    det = c["office_occupancy_detector"]
    return [
        (space, "thermal.C_air", 5e5, 1e4, 5e5),
        (space, "thermal.C_wall", 1e6, 1e5, 3e6),
        (wall, "C", 1e6, 1e4, 1e7),
        (space, "thermal.R_out", 0.5, 0.01, 1),
        (space, "thermal.R_in", 0.1, 0.01, 1),
        (wall, "R_a", 0.04, 1e-4, 1),
        (wall, "R_b", 0.04, 1e-4, 1),
        (space, "thermal.f_wall", 0.1, 0, 10),
        (space, "thermal.f_air", 0.1, 0, 10),
        (space, "thermal.Q_occ_gain", 100.0, 10, 200),
        (heater, "thermalMassHeatCapacity", 1e4, 1e3, 2e5),
        (heater, "UA", None, 1, 100),
        (hc, "kp", 0.005, 1e-5, 1, "private"),
        (cc, "kp", 0.0001, 1e-5, 1, "private"),
        ([hc, cc], "Ti", 30, 1, 300, "private"),
        ([hc, cc], "Td", 0, 0, 1, "private"),
        (valve, "waterFlowRateMax", 0.001, 1e-6, 0.1),
        (valve, "valveAuthority", 1, 0.4, 1),
        ([sup, occ.supply_damper], "a", 1, 1, 10, "shared"),
        ([sup, occ.supply_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([exh, occ.exhaust_damper], "a", 1, 1, 10, "shared"),
        ([exh, occ.exhaust_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([space, occ], "mass.V", 65, 50, 80, "shared"),
        ([space, occ], "mass.G_occ", 1e-6, 1e-6, 1e-5, "shared"),
        ([space, occ], "mass.m_inf", 0.001, 1e-4, 0.01, "shared"),
        (det, "threshold", 1.0, 0.02, 5.0),
    ]


def build_measurements(model):
    return [(model.components["office_valve_position_sensor"], 0.05 / 2),
            (model.components["office_temperature_sensor"], 0.1 / 2),
            (model.components["office_damper_position_sensor"], 0.05 / 2),
            (model.components["office_co2_sensor"], 30 / 2)]


COLLOC_OPTS = {"boundary_state_init": "rollout", "early_stopping": False}
print("model builders ready")

In [ ]:
# Small on purpose: 1 day, not 5.  Capture validity depends on WHICH ops run,
# not how many segments they run over.
import datetime

from dateutil import tz

START = [datetime.datetime(2023, 12, 2, tzinfo=tz.gettz("Europe/Copenhagen"))]
END = [datetime.datetime(2023, 12, 3, tzinfo=tz.gettz("Europe/Copenhagen"))]

if "cuda" not in DEVICES:
    raise RuntimeError("This notebook needs a GPU: Runtime > Change runtime type > GPU")

import os

os.environ["TWIN4BUILD_GRAPH_DEBUG"] = "1"
# CRITICAL: the graph is OFF for this solve.  A failed capture can leave the
# CUDA context unable to capture ANYTHING afterwards -- every later attempt then
# reports "operation failed due to a previous error during capture", which reads
# as six independent failures when it is one failure and five echoes.  The first
# capture attempt in this process must be the diagnostic's own.
os.environ["TWIN4BUILD_CUDA_GRAPH"] = "0"

from twin4build.estimator import _cuda_graph as cg

cg.DEBUG_PARTS.clear()
cg.CAPTURE_LOG.clear()

model = build_model("cuda", tag="diag")
est = tb.Estimator(tb.Simulator(model))
opts = dict(COLLOC_OPTS)
opts.update({"maxiter": 2, "exact_hessian": True})
est.estimate(START, END, STEP, build_parameters(model), build_measurements(model),
             n_warmup=N_WARMUP, method=("casadi", "ipopt", "ad", "collocation"),
             options=opts)

print()
print("capture attempts during the solve (should be NONE):")
for line in cg.capture_status() or ["  (none -- correct)"]:
    print("  " + str(line))
print("pieces available:", sorted(k for k in cg.DEBUG_PARTS))
print("problem shape   :", cg.DEBUG_PARTS.get("shapes"))

In [ ]:
import torch

inp = cg.DEBUG_PARTS.get("inputs")
if inp is None:
    raise RuntimeError("no inputs recorded -- the Hessian was never called")
print("real inputs recorded:")
for k, v in inp.items():
    print(f"  {k:12s} {tuple(v.shape)} {v.dtype}")

DEV = inp["y_norm"].device
_a = torch.randn(64, 64, dtype=torch.float64, device=DEV)


def canary():
    """A trivially capturable graph.  If THIS fails, the context is poisoned
    and every result after that point is meaningless rather than negative."""
    try:
        s = torch.cuda.Stream()
        s.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(s):
            for _ in range(3):
                _a @ _a
        torch.cuda.current_stream().wait_stream(s)
        torch.cuda.synchronize()
        g = torch.cuda.CUDAGraph()
        with torch.cuda.graph(g):
            _ = _a @ _a
        g.replay()
        torch.cuda.synchronize()
        return True
    except Exception:
        return False


def try_capture(fn, kwargs, mode):
    try:
        s = torch.cuda.Stream()
        s.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(s):
            for _ in range(3):
                fn(**kwargs)
        torch.cuda.current_stream().wait_stream(s)
        torch.cuda.synchronize()
        g = torch.cuda.CUDAGraph()
        with torch.cuda.graph(g, capture_error_mode=mode):
            fn(**kwargs)
        g.replay()
        torch.cuda.synchronize()
        return True, "captured"
    except Exception as exc:
        return False, f"{type(exc).__name__}: {str(exc).strip().splitlines()[0][:100]}"


print()
print("=" * 72)
print("CAPTURE BISECTION (canary-checked)")
print("=" * 72)
if not canary():
    raise RuntimeError(
        "The canary failed BEFORE any real attempt -- the CUDA context cannot "
        "capture at all in this session. Restart the runtime and re-run; do "
        "not read the results below."
    )
print("  canary: OK -- the context can capture, so failures below are real")
print()

n_seg = cg.DEBUG_PARTS["shapes"]["n_seg"]
_w = torch.zeros((n_seg, inp["Jt_"].shape[1]), dtype=inp["y_norm"].dtype, device=DEV)
PIECES = [
    ("curvature (constraints only)", cg.DEBUG_PARTS["curvature"],
     {"theta_norm": inp["theta_norm"], "y_norm": inp["y_norm"],
      "lam_mat": inp["lam_mat"]}),
    ("obj_curvature (+ composer.F_aug)", cg.DEBUG_PARTS["obj_curvature"],
     {"theta_norm": inp["theta_norm"], "y_norm": inp["y_norm"], "w_mat": _w}),
    ("hess_core (everything)", cg.DEBUG_PARTS["core"], dict(inp)),
]

results, poisoned_at = {}, None
for label, fn, kwargs in PIECES:
    for mode in ("global", "relaxed"):
        ok, msg = try_capture(fn, kwargs, mode)
        alive = canary()          # did this attempt poison the context?
        results[(label, mode)] = (ok, alive)
        flag = "" if alive else "   <-- CONTEXT POISONED; later rows are void"
        print(f"  {label:34s} {mode:8s} {'OK  ' if ok else 'FAIL'}  {msg}{flag}")
        if not alive and poisoned_at is None:
            poisoned_at = (label, mode)
    if poisoned_at:
        break

print()
print("=" * 72)
print("VERDICT")
print("=" * 72)
if poisoned_at:
    print(f"  The context stopped being able to capture after: {poisoned_at[0]}")
    print("  ({poisoned} mode).  Only rows ABOVE that line are evidence."
          .format(poisoned=poisoned_at[1]))
    print("  To test the remaining pieces, restart the runtime and reorder")
    print("  PIECES so the untested one runs first.")
    print()

c = results.get(("curvature (constraints only)", "global"), (None, None))[0]
o = results.get(("obj_curvature (+ composer.F_aug)", "global"), (None, None))[0]
k = results.get(("hess_core (everything)", "global"), (None, None))[0]
rel = any(ok for (lbl, m), (ok, _al) in results.items() if m == "relaxed")

if c is False and o is None:
    print("  The SIMPLEST piece -- vmap(hessian(...)) with no composer -- already")
    print("  fails on a context proven able to capture.")
    print("  => torch.func transforms are not capturable here.  That is a")
    print("     property of the transform machinery, not of this model, and")
    print("     CUDA graphs are not viable for this region.")
elif c and o is False:
    print("  Constraint curvature captures, objective curvature does not.")
    print("  => the offender is inside composer.F_aug (a component forward).")
elif c and o and k is False:
    print("  Both halves capture, the whole does not -> the einsum/packing glue.")
elif k:
    print("  Everything captures.  Then the failure in the full 5-day run is")
    print("  size-dependent -- suspect allocation, not an illegal op.")
if rel:
    print()
    print("  relaxed mode captured something global could not: the cause is a")
    print("  SYNCHRONISATION, and capture_error_mode='relaxed' may be the fix.")